In [1]:
#!/usr/bin/env python3

import pandas as pd
import networkx as nx
from pathlib import Path

# =========================
# CONFIG
# =========================

INPUT_FOLDER = Path(r"C:/Users/ak4086/Desktop/Data TEMET/output")

COREF_FILE = INPUT_FOLDER / "sir_ref_coref_edgelist.csv"
COAUTHOR_FILE = INPUT_FOLDER / "sir_ref_coauthor_edgelist.csv"

OUTPUT_FILE = INPUT_FOLDER / "sir_ref_network_metrics.csv"


# =========================
# FUNCTIONS
# =========================

def load_graph_from_edgelist(file_path, network_name):
    df = pd.read_csv(file_path)

    G = nx.Graph()

    for _, row in df.iterrows():
        source = row["source"]
        target = row["target"]

        weight = row["weight"] if "weight" in df.columns else 1

        G.add_edge(source, target, weight=weight)

    return G


def calculate_network_metrics(G, network_name):
    if G.number_of_nodes() == 0:
        return {
            "Network": network_name,
            "Nodes": 0,
            "Edges": 0,
            "Clustering_Coefficient": 0,
            "Avg_Degree_Centrality": 0,
            "Avg_Betweenness_Centrality": 0
        }

    clustering_coefficient = nx.average_clustering(G)

    degree_centrality = nx.degree_centrality(G)
    avg_degree_centrality = (
        sum(degree_centrality.values()) / len(degree_centrality)
        if degree_centrality else 0
    )

    if G.number_of_nodes() > 1:
        betweenness_centrality = nx.betweenness_centrality(G)
        avg_betweenness_centrality = (
            sum(betweenness_centrality.values()) / len(betweenness_centrality)
        )
    else:
        avg_betweenness_centrality = 0

    return {
        "Network": network_name,
        "Nodes": G.number_of_nodes(),
        "Edges": G.number_of_edges(),
        "Clustering_Coefficient": clustering_coefficient,
        "Avg_Degree_Centrality": avg_degree_centrality,
        "Avg_Betweenness_Centrality": avg_betweenness_centrality
    }


# =========================
# RUN
# =========================

coref_graph = load_graph_from_edgelist(COREF_FILE, "Coreference")
coauthor_graph = load_graph_from_edgelist(COAUTHOR_FILE, "Coauthorship")

results = [
    calculate_network_metrics(coref_graph, "Coreference"),
    calculate_network_metrics(coauthor_graph, "Coauthorship")
]

df_results = pd.DataFrame(results)

df_results.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(df_results)
print(f"\nSaved metrics to: {OUTPUT_FILE}")

        Network  Nodes  Edges  Clustering_Coefficient  Avg_Degree_Centrality  \
0   Coreference     47    212                0.604662               0.196115   
1  Coauthorship    138    271                0.779360               0.028668   

   Avg_Betweenness_Centrality  
0                    0.029561  
1                    0.000220  

Saved metrics to: C:\Users\ak4086\Desktop\Data TEMET\output\sir_ref_network_metrics.csv
